In [1]:
%pip install --upgrade huggingface-hub
%pip install opencv-python scipy tqdm lmdb yapf tb-nightly torchvision
%pip install git+https://github.com/ai-forever/Real-ESRGAN.git

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/ai-forever/Real-ESRGAN.git to C:\Users\himan\AppData\Local\Temp\pip-req-build-2wu77b0x
  Resolved https://github.com/ai-forever/Real-ESRGAN.git to commit 362a0316878f41dbdfbb23657b450c3353de5acf
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/ai-forever/Real-ESRGAN.git 'C:\Users\himan\AppData\Local\Temp\pip-req-build-2wu77b0x'
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      Traceback (most recent call last):
        File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return ho

In [2]:
'''
import torch
import gradio as gr
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# Load model globally to avoid reloading on every click
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cache = {}

def get_model(scale):
    if scale not in model_cache:
        model = RealESRGANer(
            scale=scale,
            model_path=f'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x{scale}.pth',
            upsampler=RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale),
            tile=400,
            tile_pad=10,
            pre_pad=0,
            half=False
        )
        model_cache[scale] = model
    return model_cache[scale]

def upscale_ui(input_img, scale):
    if input_img is None:
        return None
    
    # Initialize/Get model
    model = get_model(int(scale))
    
    # Process image
    sr_image, _ = model.enhance(input_img, outscale=int(scale))
    return sr_image

# Build the Gradio Interface
interface = gr.Interface(
    fn=upscale_ui,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Radio([2, 4, 8], label="Upscale Factor", value=4)
    ],
    outputs=gr.Image(type="pil", label="Upscaled Result"),
    title="Real-ESRGAN Image Upscaler",
    description="Upload a blurry or low-res image to enhance it using AI.",
    allow_flagging="never"
)

if __name__ == "__main__":
    # This will launch a local web server
    interface.launch()
'''

'\nimport torch\nimport gradio as gr\nfrom PIL import Image\nfrom realesrgan import RealESRGANer\nfrom basicsr.archs.rrdbnet_arch import RRDBNet\n\n# Load model globally to avoid reloading on every click\ndevice = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\nmodel_cache = {}\n\ndef get_model(scale):\n    if scale not in model_cache:\n        model = RealESRGANer(\n            scale=scale,\n            model_path=f\'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x{scale}.pth\',\n            upsampler=RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale),\n            tile=400,\n            tile_pad=10,\n            pre_pad=0,\n            half=False\n        )\n        model_cache[scale] = model\n    return model_cache[scale]\n\ndef upscale_ui(input_img, scale):\n    if input_img is None:\n        return None\n\n    # Initialize/Get model\n    model = get_model(int(scale))\n\n    # Process image\n    s

In [3]:
!pip install torch torchvision torchaudio

In [2]:
import torch
import gradio as gr
import os
import numpy as np
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cache = {}

def get_model(scale):
    if scale not in model_cache:
        model = RealESRGANer(
            scale=scale,
            model_path=f'weights/RealESRGAN_x{scale}.pth',
            model=RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale),
            tile=400,
            tile_pad=10,
            pre_pad=0,
            half=False
        )
        model_cache[scale] = model
    return model_cache[scale]

def batch_upscale(images, scale):
    if not images:
        return None
    scale = int(scale)
    model = get_model(scale)
    upscaled_list = []
    
    for img_path in images:
        # Gradio 'files' input returns paths; load them as PIL images
        img = Image.open(img_path).convert('RGB')
        img = np.array(img)
        sr_img, _ = model.enhance(img, outscale=scale)
        upscaled_list.append(sr_img)
        
    return upscaled_list

# Define the UI
with gr.Blocks(title="Batch Real-ESRGAN") as demo:
    gr.Markdown("# 🚀 Batch AI Image Upscaler")
    gr.Markdown("Upload multiple images to upscale them simultaneously using Real-ESRGAN.")
    
    with gr.Row():
        with gr.Column():
            # File input allows multiple selection
            file_input = gr.File(file_count="multiple", label="Upload Images", file_types=["image"])
            scale_input = gr.Radio([2, 4, 8], label="Upscale Factor", value=4)
            process_btn = gr.Button("Start Batch Processing", variant="primary")
            
        with gr.Column():
            # Gallery displays the results
            gallery_output = gr.Gallery(label="Upscaled Results", columns=2, height="auto")

    process_btn.click(
        fn=batch_upscale, 
        inputs=[file_input, scale_input], 
        outputs=gallery_output
    )

if __name__ == "__main__":
    demo.launch()


ModuleNotFoundError: No module named 'torchvision.transforms.functional_tensor'

In [3]:
import torch
import gradio as gr
import os
import zipfile
import io
import numpy as np
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# --- Helper to load model ---
def load_model(scale, device_type):
    device = torch.device(device_type)
    model = RealESRGANer(
        scale=scale,
        model_path=f'weights/RealESRGAN_x{scale}.pth',
        model=RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale),
        tile=400,
        tile_pad=10,
        pre_pad=0,
        half=False,
        device=device
    )
    return model

# --- Main Batch Function ---
def batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, None
    
    scale = int(scale)
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for i, img_path in enumerate(progress.tqdm(images, desc="Upscaling Batch")):
            filename = os.path.basename(img_path)
            try:
                device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
                model = load_model(scale, device_name)
                
                img = Image.open(img_path).convert('RGB')
                img = np.array(img)
                sr_img, _ = model.enhance(img, outscale=scale)
                upscaled_images.append(sr_img)
                
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename}", img_byte_arr.getvalue())
                
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception as e:
                print(f"Error on {filename}: {e}")
                continue

    zip_buffer.seek(0)
    zip_path = "upscaled_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getvalue())
    
    return zip_path, upscaled_images

# --- UI Setup ---
with gr.Blocks() as demo:
    gr.Markdown("# 🛰️ ISRO/SVNIT Image Enhancement Tool")
    gr.Markdown("Uses Real-ESRGAN with **Tiling** to prevent Memory Errors.")
    
    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Upload Images")
            scale_input = gr.Radio([2, 4, 8], label="Scale Factor", value=4)
            btn = gr.Button("Start Processing", variant="primary")
        with gr.Column():
            zip_output = gr.File(label="Download ZIP Archive")
            gallery_output = gr.Gallery(label="Preview Results", columns=2)

    btn.click(batch_upscale, [file_input, scale_input], [zip_output, gallery_output])


if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


e:\Research\svnit isro\implement\.venv\Lib\site-packages\realesrgan\utils.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loadnet = torch.load(model_path, map_location=

Error on 2025-12-07_210312_SR_preview.png: 'params'


In [1]:
import torch
import gradio as gr
import os
import zipfile
import io
from PIL import Image
from realESRGAN import RealESRGANer

# --- Fixed Model Loader ---
def load_model(scale, device_type):
    device = torch.device(device_type)
    model = RealESRGANer(device, scale=int(scale))
    model.load_weights(f'weights/RealESRGAN_x{scale}.pth', download=True)
    return model

# --- Fixed Batch Function ---
def batch_upscale_with_zip(images, scale):
    if not images:
        return None, None
    
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in images:
            filename = os.path.basename(img_path)
            try:
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
                model = load_model(scale, device)
                
                img = Image.open(img_path).convert('RGB')
                
                # --- OOM PREVENTION STRATEGY ---
                # 1. Use tile to process image in patches
                # 2. Use half=True to use FP16 (if your GPU supports it)
                try:
                    sr_img = model.predict(
                        img, 
                        tile=512,        # Processes 512x512 chunks
                        tile_pad=10,     # Overlap to prevent seams
                        half=True        # Use half-precision (save 50% VRAM)
                    )
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower():
                        print(f"Still OOM on GPU, switching {filename} to CPU...")
                        torch.cuda.empty_cache()
                        model = load_model(scale, 'cpu')
                        sr_img = model.predict(img) # CPU doesn't usually need tiling
                    else:
                        raise e

                upscaled_images.append(sr_img)
                
                # Save to ZIP logic...
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename}", img_byte_arr.getvalue())
                
                del model
                torch.cuda.empty_cache()

            except Exception as e:
                print(f"Error on {filename}: {e}")
                continue

    zip_buffer.seek(0)
    # ... return logic ...

    # Finalize ZIP file
    zip_buffer.seek(0)
    zip_filename = "upscaled_batch.zip"
    with open(zip_filename, "wb") as f:
        f.write(zip_buffer.read())
    
    return zip_filename, upscaled_images

# --- UI Setup ---
with gr.Blocks() as demo:
    gr.Markdown("# Real-ESRGAN Batch Processor")
    with gr.Row():
        with gr.Column():
            inputs = gr.File(file_count="multiple", label="Select Images")
            scale = gr.Radio([2, 4, 8], label="Scale Factor", value=4)
            btn = gr.Button("Upscale All")
        with gr.Column():
            zip_out = gr.File(label="Download ZIP")
            gallery = gr.Gallery(label="Preview")

    btn.click(batch_upscale_with_zip, [inputs, scale], [zip_out, gallery])

demo.launch()

ModuleNotFoundError: No module named 'realESRGAN'

In [3]:
import torch
import gradio as gr
import os
import zipfile
import io
import numpy as np
from PIL import Image
from realesrgan import RealESRGANer

# --- Model Loader ---
def load_model(scale, device_type):
    device = torch.device(device_type)
    model = RealESRGANer(device, scale=int(scale))
    model.load_weights(f'weights/RealESRGANer_x{scale}.pth', download=True)
    return model

# --- Manual Tiling Function ---
def upscale_with_manual_tiling(model, PIL_img, scale, tile_size=512):
    """Splits image into tiles, upscales, and merges them."""
    w, h = PIL_img.size
    # Create a new blank image for the output
    new_w, new_h = w * scale, h * scale
    output_img = Image.new('RGB', (new_w, new_h))

    for y in range(0, h, tile_size):
        for x in range(0, w, tile_size):
            # Define the tile box
            box = (x, y, min(x + tile_size, w), min(y + tile_size, h))
            tile = PIL_img.crop(box)
            
            # Upscale the small tile
            upscaled_tile = model.predict(tile)
            
            # Paste it into the correct position in the output
            output_img.paste(upscaled_tile, (x * scale, y * scale))
            
    return output_img

# --- Main Batch Function ---
def batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, None
    
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc="Processing Batch"):
            filename = os.path.basename(img_path)
            try:
                device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
                model = load_model(scale, device_name)
                img = Image.open(img_path).convert('RGB')
                
                # Use manual tiling to prevent OOM
                sr_img = upscale_with_manual_tiling(model, img, int(scale))
                
                upscaled_images.append(sr_img)
                
                # Save to ZIP
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename}", img_byte_arr.getvalue())
                
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"Error on {filename}: {e}")
                continue

    zip_buffer.seek(0)
    zip_path = "upscaled_batch_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getbuffer())
    
    return zip_path, upscaled_images

# --- UI Setup ---
with gr.Blocks() as demo:
    gr.Markdown("### 🛰️ Robust Batch Upscaler (Manual Tiling Mode)")
    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Upload Images")
            scale_input = gr.Radio([2, 4, 8], label="Scale Factor", value=4)
            btn = gr.Button("Upscale All", variant="primary")
        with gr.Column():
            zip_output = gr.File(label="Download ZIP")
            gallery_output = gr.Gallery(label="Preview")

    btn.click(batch_upscale, [file_input, scale_input], [zip_output, gallery_output])

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Error on 20251220_220921.jpg: RealESRGANer.__init__() got multiple values for argument 'scale'


In [ ]:
import torch
import gradio as gr
import os
import zipfile
import io
from PIL import Image
from RealESRGAN import RealESRGAN

# --- Model Loader ---
def load_model(scale, device_type):
    device = torch.device(device_type)
    model = RealESRGAN(device, scale=int(scale))
    model.load_weights(f'weights/RealESRGAN_x{scale}.pth', download=True)
    return model

# --- Seamless Manual Tiling ---
def upscale_seamless(model, PIL_img, scale, tile_size=512, padding=10):
    """Upscales with overlapping tiles to prevent visible seams."""
    w, h = PIL_img.size
    new_w, new_h = w * scale, h * scale
    output_img = Image.new('RGB', (new_w, new_h))

    # Iterate through the image with a step size that accounts for padding
    for y in range(0, h, tile_size - padding):
        for x in range(0, w, tile_size - padding):
            # Define crop area
            box = (x, y, min(x + tile_size, w), min(y + tile_size, h))
            tile = PIL_img.crop(box)
            
            # Process tile
            upscaled_tile = model.predict(tile)
            
            # Calculate paste position
            output_img.paste(upscaled_tile, (x * scale, y * scale))
            
    return output_img

# --- Main Batch Function with Progress Bar ---
def batch_upscale_manager(images, scale, progress=gr.Progress(track_tqdm=True)):
    if not images:
        return None, None
    
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    # Initialize ZIP
    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        # progress.tqdm creates the progress bar in the Gradio UI
        for img_path in progress.tqdm(images, desc="Enhancing Images..."):
            filename = os.path.basename(img_path)
            
            try:
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
                model = load_model(scale, device)
                
                img = Image.open(img_path).convert('RGB')
                
                # Execute seamless tiling
                sr_img = upscale_seamless(model, img, int(scale))
                
                upscaled_images.append(sr_img)
                
                # Save to ZIP buffer
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename.split('.')[0]}.png", img_byte_arr.getvalue())
                
                # Memory Management
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
                continue

    # Finalize ZIP
    zip_buffer.seek(0)
    zip_filename = "batch_upscale_results.zip"
    with open(zip_filename, "wb") as f:
        f.write(zip_buffer.getvalue())
    
    return zip_filename, upscaled_images

# --- Gradio UI Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## ⚡ Real-ESRGAN Research Batch Tool")
    gr.Markdown("Optimized for SVNIT/ISRO research: Includes **Manual Tiling** for OOM prevention and **Seamless Blending**.")
    
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(file_count="multiple", label="Select Images", file_types=["image"])
            scale_input = gr.Radio([2, 4, 8], label="Upscale Factor", value=4)
            process_btn = gr.Button("🚀 Start Batch Process", variant="primary")
            
        with gr.Column(scale=2):
            zip_output = gr.File(label="📦 Download All Results (ZIP)")
            gallery_output = gr.Gallery(label="✨ Preview Results", columns=2, object_fit="contain")

    process_btn.click(
        fn=batch_upscale_manager, 
        inputs=[file_input, scale_input], 
        outputs=[zip_output, gallery_output]
    )

if __name__ == "__main__":
    demo.launch()

Fater model

In [ ]:
import torch
import gradio as gr
import os
import zipfile
import io
from PIL import Image
from RealESRGAN import RealESRGAN

# --- Optimized Model Loader ---
def load_fast_model(scale, device_type):
    device = torch.device(device_type)
    model = RealESRGAN(device, scale=int(scale))
    model.load_weights(f'weights/RealESRGAN_x{scale}.pth', download=True)
    
    if device_type == 'cuda':
        model.model.half() # Convert model weights to FP16
    return model

# --- Fast Seamless Tiling ---
def fast_tile_process(model, img, scale, device):
    # If image is small (e.g., < 800px), process directly
    if max(img.size) < 800:
        with torch.amp.autocast(device_type=device, enabled=(device == 'cuda')):
            return model.predict(img)
    
    # Otherwise, use tiling to prevent CUDNN_STATUS_EXECUTION_FAILED
    w, h = img.size
    tile_size = 512
    new_w, new_h = w * scale, h * scale
    output_img = Image.new('RGB', (new_w, new_h))

    for y in range(0, h, tile_size - 10):
        for x in range(0, w, tile_size - 10):
            box = (x, y, min(x + tile_size, w), min(y + tile_size, h))
            tile = img.crop(box)
            with torch.amp.autocast(device_type=device, enabled=(device == 'cuda')):
                upscaled_tile = model.predict(tile)
            output_img.paste(upscaled_tile, (x * scale, y * scale))
    return output_img

def fast_batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, None
    
    upscaled_images = []
    zip_buffer = io.BytesIO()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    model = load_fast_model(scale, device)
    
    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc="Turbo Processing"):
            filename = os.path.basename(img_path)
            try:
                img = Image.open(img_path).convert('RGB')
                
                # Apply fast tiling logic
                sr_img = fast_tile_process(model, img, int(scale), device)
                
                upscaled_images.append(sr_img)
                
                # Save as JPEG for speed
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='JPEG', quality=90)
                zip_file.writestr(f"fast_{filename}", img_byte_arr.getvalue())

            except Exception as e:
                print(f"Error on {filename}: {e}")
                continue
            finally:
                if device == 'cuda':
                    torch.cuda.empty_cache()

    zip_buffer.seek(0)
    zip_path = "fast_upscale_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getvalue())
    
    return zip_path, upscaled_images

# --- UI Setup ---
with gr.Blocks(theme=gr.themes.Default()) as demo:
    gr.Markdown("## ⚡ Fast Real-ESRGAN (Research Optimized)")
    with gr.Row():
        with gr.Column():
            files = gr.File(file_count="multiple", label="Input Images")
            scale = gr.Radio([2, 4], label="Scale", value=4)
            run = gr.Button("🚀 Run Fast Process", variant="primary")
        with gr.Column():
            out_zip = gr.File(label="Download ZIP")
            gallery = gr.Gallery(label="Previews", columns=3)

    run.click(fast_batch_upscale, [files, scale], [out_zip, gallery])

demo.launch()

In [ ]:
import torch
import gradio as gr
import os
import zipfile
import io
from PIL import Image
from RealESRGAN import RealESRGAN

# --- Model Loader ---
# Moved outside the loop to prevent redundant loading
def load_model(scale, device_type):
    device = torch.device(device_type)
    model = RealESRGAN(device, scale=int(scale))
    model.load_weights(f'weights/RealESRGAN_x{scale}.pth', download=True)
    return model

# --- Seamless Manual Tiling ---
def upscale_seamless(model, PIL_img, scale, tile_size=512, padding=10):
    w, h = PIL_img.size
    new_w, new_h = w * scale, h * scale
    output_img = Image.new('RGB', (new_w, new_h))

    # Tiling helps process large images without crashing GPU memory (VRAM)
    for y in range(0, h, tile_size - padding):
        for x in range(0, w, tile_size - padding):
            # Define crop area
            box = (x, y, min(x + tile_size, w), min(y + tile_size, h))
            tile = PIL_img.crop(box)
            
            # Process tile
            upscaled_tile = model.predict(tile)
            
            # Calculate paste position
            output_img.paste(upscaled_tile, (x * scale, y * scale))
            
    return output_img

# --- Main Batch Function ---
def batch_upscale_manager(images, scale, progress=gr.Progress(track_tqdm=True)):
    if not images:
        return None, None
    
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    # 1. Load model ONCE before starting the loop
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = load_model(scale, device)
    
    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc="Enhancing Images..."):
            filename = os.path.basename(img_path)
            
            try:
                img = Image.open(img_path).convert('RGB')
                
                # 2. Execute tiling
                sr_img = upscale_seamless(model, img, int(scale))
                upscaled_images.append(sr_img)
                
                # 3. Save to ZIP
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{os.path.splitext(filename)[0]}.png", img_byte_arr.getvalue())
                
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
                continue

    # Clean up memory after the whole batch is done
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    zip_buffer.seek(0)
    zip_filename = "batch_results.zip"
    with open(zip_filename, "wb") as f:
        f.write(zip_buffer.getvalue())
    
    return zip_filename, upscaled_images

# --- Gradio UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## ⚡ Real-ESRGAN Research Batch Tool")
    
    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Upload Images")
            scale_input = gr.Radio([2, 4, 8], label="Scale", value=4)
            process_btn = gr.Button("🚀 Start Process", variant="primary")
            
        with gr.Column():
            zip_output = gr.File(label="📦 Download ZIP")
            gallery_output = gr.Gallery(label="Preview", columns=2)

    process_btn.click(
        fn=batch_upscale_manager, 
        inputs=[file_input, scale_input], 
        outputs=[zip_output, gallery_output]
    )

if __name__ == "__main__":
    demo.launch()

ModuleNotFoundError: No module named 'RealESRGAN'

In [1]:
import torch
import numpy as np
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
import gradio as gr

# Use CUDA if available; otherwise fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# For an NVIDIA RTX 3050 4GB, smaller tile sizes and FP16 help reduce VRAM use.
GPU_TILE_SIZE = 256

gpu_model_cache = {}

def get_gpu_model(scale):
    scale = int(scale)
    if scale not in gpu_model_cache:
        gpu_model_cache[scale] = RealESRGANer(
            scale=scale,
            model_path=f"weights/RealESRGAN_x{scale}.pth",
            model=RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale),
            tile=GPU_TILE_SIZE,
            tile_pad=10,
            pre_pad=0,
            half=True,
            device=device,
        )
    return gpu_model_cache[scale]


def batch_upscale_gpu(images, scale):
    if not images:
        return None

    scale = int(scale)
    model = get_gpu_model(scale)
    results = []

    for img_path in images:
        img = Image.open(img_path).convert("RGB")
        img_array = np.array(img)
        sr_img, _ = model.enhance(img_array, outscale=scale)
        results.append(sr_img)

        if device.type == "cuda":
            torch.cuda.empty_cache()

    return results

# --- GPU Gradio UI ---
with gr.Blocks(title="Real-ESRGAN GPU Batch Upscaler") as demo_gpu:
    gr.Markdown("## Real-ESRGAN GPU Batch Upscaler (RTX 3050)")
    gr.Markdown("Optimized for your NVIDIA RTX 3050 4GB: smaller tiles, FP16, and GPU batching.")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Upload Images", file_types=["image"])
            scale_input = gr.Radio([2, 4], label="Scale Factor", value=4)
            process_btn = gr.Button("Upscale on GPU", variant="primary")

        with gr.Column():
            gallery_output = gr.Gallery(label="Upscaled Results", columns=2, object_fit="contain")

    process_btn.click(
        fn=batch_upscale_gpu,
        inputs=[file_input, scale_input],
        outputs=gallery_output
    )

if __name__ == "__main__":
    demo_gpu.launch()


ModuleNotFoundError: No module named 'torchvision.transforms.functional_tensor'

In [ ]:
import torch
import subprocess
import sys

print("=" * 60)
print("GPU DIAGNOSTICS FOR RTX 3050")
print("=" * 60)

# Check CUDA availability
print(f"\n✓ CUDA Available: {torch.cuda.is_available()}")
print(f"✓ CUDA Version (PyTorch): {torch.version.cuda}")
print(f"✓ PyTorch Version: {torch.__version__}")

if torch.cuda.is_available():
    print(f"✓ GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
    print(f"✓ Current Device: {torch.cuda.current_device()}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️  CUDA NOT DETECTED - GPU WILL NOT BE USED")
    print("\nTo enable GPU support for RTX 3050, run this:")
    print("-" * 60)
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
    print("-" * 60)
    print("\nOr for CUDA 12.1:")
    print("-" * 60)
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
    print("-" * 60)

print("\n" + "=" * 60)


GPU DIAGNOSTICS FOR RTX 3050

✓ CUDA Available: False
✓ CUDA Version (PyTorch): None
✓ PyTorch Version: 2.11.0+cpu

⚠️  CUDA NOT DETECTED - GPU WILL NOT BE USED

To enable GPU support for RTX 3050, run this:
------------------------------------------------------------
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
------------------------------------------------------------

Or for CUDA 12.1:
------------------------------------------------------------
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
------------------------------------------------------------



In [1]:
import torch
import gradio as gr
import os
import zipfile
import io
import numpy as np
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# --- Optimized Model Loader ---
def load_model(scale, device_type):
    # 1. Define the architecture (RRDBNet is the standard for Real-ESRGAN)
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=int(scale))
    
    # 2. Path to your weights
    model_path = f'weights/RealESRGAN_x{scale}.pth' 
    
    # 3. Initialize the upsampler (RealESRGANer handles tiling and device automatically)
    upsampler = RealESRGANer(
        scale=int(scale),
        model_path=model_path,
        model=model,
        tile=512,           # Automatically split image into 512x512 tiles to save memory
        tile_pad=10,        # Overlap to ensure seamless merging
        pre_pad=0,
        half=(device_type == 'cuda'), # Use FP16 on GPU (saves 50% VRAM)
        device=torch.device(device_type)
    )
    return upsampler

# --- Main Batch Function ---
def batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, None
    
    scale = int(scale)
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    # Load model once for the whole batch
    device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
    try:
        model = load_model(scale, device_name)
    except Exception as e:
        return None, f"Error loading model: {e}"

    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc="Processing Batch"):
            filename = os.path.basename(img_path)
            try:
                # Load image
                img = Image.open(img_path).convert('RGB')
                img_np = np.array(img)
                
                # Enhance image (Official library uses .enhance instead of .predict)
                # This automatically handles the tiling we configured in load_model
                sr_img_np, _ = model.enhance(img_np, outscale=scale)
                sr_img = Image.fromarray(sr_img_np)
                
                upscaled_images.append(sr_img)
                
                # Save to ZIP
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename.split('.')[0]}.png", img_byte_arr.getvalue())

            except Exception as e:
                print(f"Error on {filename}: {e}")
                continue

    # Finalize ZIP
    zip_buffer.seek(0)
    zip_path = "upscaled_batch_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getbuffer())
    
    return zip_path, upscaled_images

# --- UI Setup ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("### 🛰️ ISRO/SVNIT High-Resolution Image Enhancement")
    gr.Markdown("Using official Real-ESRGAN with **Automatic Tiling** and **FP16** support.")
    
    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Upload Images")
            scale_input = gr.Radio([2, 4, 8], label="Scale Factor", value=4)
            btn = gr.Button("🚀 Start Processing", variant="primary")
        with gr.Column():
            zip_output = gr.File(label="Download ZIP Archive")
            gallery_output = gr.Gallery(label="Preview Results", columns=2)

    btn.click(batch_upscale, [file_input, scale_input], [zip_output, gallery_output])

if __name__ == "__main__":
    demo.launch()


C:\Users\himan\AppData\Local\Temp\ipykernel_29216\2214639280.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\gradio\blocks.py", line 2192, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\gradio\blocks.py", line 1955, in postprocess_data
    prediction_value = await anyio.to_thread.run_sync(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Research\svnit isro\implement\model\.venv\Lib\site-packages\anyio\to_thread.py", l

In [ ]:
import torch
import gradio as gr
import os
import zipfile
import io
import numpy as np
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# --- Optimized Model Loader ---
def load_model(scale, device_type):
    scale = int(scale)
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale)
    
    # Path to your weights
    original_path = f'weights/RealESRGAN_x{scale}.pth'
    fixed_path = f'weights/RealESRGAN_x{scale}_fixed.pth'
    
    # --- AUTO-FIX WEIGHT FORMAT ---
    # The official library expects {'params': state_dict}. If your file is just the state_dict, we fix it.
    if not os.path.exists(fixed_path):
        if not os.path.exists(original_path):
            raise FileNotFoundError(f"Weight file not found: {original_path}")
        
        print(f"Adapting weight format for {original_path}...")
        loadnet = torch.load(original_path, map_location='cpu')
        if 'params' not in loadnet and 'params_ema' not in loadnet:
            torch.save({'params': loadnet}, fixed_path)
        else:
            fixed_path = original_path # Already in correct format
    
    # Initialize the upsampler
    upsampler = RealESRGANer(
        scale=scale,
        model_path=fixed_path,
        model=model,
        tile=512,           
        tile_pad=10,        
        pre_pad=0,
        half=(device_type == 'cuda'), 
        device=torch.device(device_type)
    )
    return upsampler

# --- Main Batch Function ---
def batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, []
    
    scale = int(scale)
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
    try:
        # Load model once for the batch to save time
        model = load_model(scale, device_name)
    except Exception as e:
        raise gr.Error(f"Model Initialization Error: {e}")

    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc="Enhancing Batch"):
            filename = os.path.basename(img_path)
            try:
                # 1. Load and convert to RGB
                img = Image.open(img_path).convert('RGB')
                img_np = np.array(img)
                
                # 2. Process with automatic tiling
                sr_img_np, _ = model.enhance(img_np, outscale=scale)
                sr_img = Image.fromarray(sr_img_np)
                
                upscaled_images.append(sr_img)
                
                # 3. Add to ZIP
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename.split('.')[0]}.png", img_byte_arr.getvalue())

                # Clear some memory
                del img, img_np, sr_img_np
                
            except Exception as e:
                print(f"Error on {filename}: {e}")
                continue

    if not upscaled_images:
        raise gr.Error("All images failed to process. Check console for details.")

    # Finalize ZIP
    zip_buffer.seek(0)
    zip_path = "upscaled_batch_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getbuffer())
    
    # Clean up GPU memory after batch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return zip_path, upscaled_images

# --- UI Setup ---
with gr.Blocks(theme=gr.themes.Soft(), title="SVNIT/ISRO Image Enhancer") as demo:
    gr.Markdown("### 🛰️ Real-ESRGAN Research Batch Tool")
    gr.Markdown("Optimized for high-resolution imagery. Features **Automatic Tiling** and **VRAM Optimization**.")
    
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(file_count="multiple", label="Input Images")
            scale_input = gr.Radio([2, 4, 8], label="Scale Factor", value=4)
            btn = gr.Button("🚀 Start Processing", variant="primary")
        with gr.Column(scale=2):
            zip_output = gr.File(label="Download ZIP Results")
            gallery_output = gr.Gallery(label="Preview", columns=2, object_fit="contain")

    btn.click(batch_upscale, [file_input, scale_input], [zip_output, gallery_output])

if __name__ == "__main__":
    demo.launch()


In [1]:
import torch
import gradio as gr
import os
import zipfile
import io
import numpy as np
import sys
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# --- GPU Optimized Model Loader ---
def load_model(scale):
    scale = int(scale)
    # Force CUDA if available, otherwise fallback to CPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale)
    
    original_path = f'weights/RealESRGAN_x{scale}.pth'
    fixed_path = f'weights/RealESRGAN_x{scale}_fixed.pth'
    
    # Auto-fix format if needed
    if not os.path.exists(fixed_path):
        if not os.path.exists(original_path):
            raise FileNotFoundError(f"Missing weights: {original_path}")
        loadnet = torch.load(original_path, map_location='cpu')
        if 'params' not in loadnet and 'params_ema' not in loadnet:
            torch.save({'params': loadnet}, fixed_path)
        else:
            fixed_path = original_path

    upsampler = RealESRGANer(
        scale=scale,
        model_path=fixed_path,
        model=model,
        tile=400,           
        tile_pad=10,
        pre_pad=0,
        half=torch.cuda.is_available(), # FP16 is critical for RTX 3050 4GB
        device=device
    )
    return upsampler

# --- Main Batch Function ---
def batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, []
    
    is_cuda = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if is_cuda else "CPU (Check your Kernel!)"
    print(f"Running on: {gpu_name}")

    scale = int(scale)
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    try:
        model = load_model(scale)
    except Exception as e:
        raise gr.Error(f"Startup Error: {e}")

    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc=f"Upscaling on {gpu_name}"):
            filename = os.path.basename(img_path)
            try:
                img = Image.open(img_path).convert('RGB')
                img_np = np.array(img)
                
                sr_img_np, _ = model.enhance(img_np, outscale=scale)
                sr_img = Image.fromarray(sr_img_np)
                upscaled_images.append(sr_img)
                
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename.split('.')[0]}.png", img_byte_arr.getvalue())

                del img_np, sr_img_np
                if is_cuda:
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"Error: {e}")
                continue

    zip_buffer.seek(0)
    zip_path = "upscaled_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getbuffer())
    
    return zip_path, upscaled_images

# --- UI Setup ---
with gr.Blocks() as demo:
    gr.Markdown(f"### 🛰️ ISRO/SVNIT High-Res Enhancer (RTX 3050 Optimized)")
    
    # Kernel Verification Alert
    if ".venv" not in sys.executable.lower():
        gr.Markdown("> [!CAUTION]\n> **WRONG KERNEL DETECTED!** Your notebook is using Global Python, not the project `.venv`. \n> \n> **To fix:** Click the kernel name (top-right of VS Code) → 'Select Kernel' → 'Python Environments' → Choose **.venv**.")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Select Images")
            scale_input = gr.Radio([2, 4, 8], label="Upscale Factor", value=4)
            btn = gr.Button("🚀 Start GPU Processing", variant="primary")
        with gr.Column():
            zip_output = gr.File(label="Download ZIP")
            gallery_output = gr.Gallery(label="Preview Results", columns=2)

    btn.click(batch_upscale, [file_input, scale_input], [zip_output, gallery_output])

if __name__ == "__main__":
    # In Gradio 6.0+, theme is passed to launch()
    demo.launch(theme=gr.themes.Soft())


ModuleNotFoundError: No module named 'realesrgan'

In [2]:
!pip install torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121 --force-reinstall


Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 3.9 MB/s eta 0:10:31
     ---------------------------------------- 0.0/2.4 GB 3.8 MB/s eta 0:10:45
     ---------------------------------------- 0.0/2.4 GB 3.8 MB/s eta 0:10:40
     ---------------------------------------- 0.0/2.4 GB 4.5 MB/s eta 0:09:04
     ---------------------------------------- 0.0/2.4 GB 4.9 MB/s eta 0:08:22
     ---------------------------------------- 0.0/2.4 GB 5.2 MB/s eta 0:07:50
     ---------------------------------------- 0.0/2.4 GB 5.4 MB/s eta 0:07:30
     ---------------------------------------- 0.0/2.4 GB 5.6 MB/s eta 0:07:12
     ---------------------------------------- 0.0/2.4 GB 5.9 MB/s eta 0:06:56
     ---------------------------------------- 0.0/2.4 GB 6.1 MB/s eta 0:06:40
     --------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.2.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatible.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.3 which is incompatible.


In [ ]:
import torch
import gradio as gr
import os
import zipfile
import io
import numpy as np
import sys
from PIL import Image
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# --- GPU / CPU Device Diagnostics ---
is_cuda = torch.cuda.is_available()
device = torch.device('cuda' if is_cuda else 'cpu')
print(f"CUDA Available: {is_cuda}")
if is_cuda:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA GPU not detected. Falling back to CPU. (To enable GPU, check if laptop is in Eco/battery-saving mode, charger is plugged in, or GPU is disabled in Device Manager).")

# --- Optimized Model Loader ---
def load_model(scale):
    scale = int(scale)
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale)
    
    # Paths are relative to the notebook's directory
    original_path = f'weights/RealESRGAN_x{scale}.pth'
    fixed_path = f'weights/RealESRGAN_x{scale}_fixed.pth'
    
    # Auto-fix weights dictionary keys structure if needed for basicsr/realesrgan
    if not os.path.exists(fixed_path):
        if not os.path.exists(original_path):
            raise FileNotFoundError(f"Missing weights file: {original_path}")
        print(f"Adapting weight format for {original_path}...")
        loadnet = torch.load(original_path, map_location='cpu')
        if 'params' not in loadnet and 'params_ema' not in loadnet:
            torch.save({'params': loadnet}, fixed_path)
        else:
            fixed_path = original_path

    # RealESRGANer handles tiling to prevent GPU VRAM Out-Of-Memory (OOM)
    upsampler = RealESRGANer(
        scale=scale,
        model_path=fixed_path,
        model=model,
        tile=400,           # Smaller tile size (400) fits well in 4GB VRAM
        tile_pad=10,
        pre_pad=0,
        half=is_cuda,       # FP16 half-precision is enabled ONLY on CUDA (crashes on CPU)
        device=device
    )
    return upsampler

# --- Main Batch Function ---
def batch_upscale(images, scale, progress=gr.Progress()):
    if not images:
        return None, []
    
    scale = int(scale)
    upscaled_images = []
    zip_buffer = io.BytesIO()
    
    try:
        model = load_model(scale)
    except Exception as e:
        raise gr.Error(f"Startup Error: {e}")

    with zipfile.ZipFile(zip_buffer, "a", zipfile.ZIP_DEFLATED) as zip_file:
        for img_path in progress.tqdm(images, desc=f"Processing on {device}"):
            filename = os.path.basename(img_path)
            try:
                img = Image.open(img_path).convert('RGB')
                img_np = np.array(img)
                
                # Perform enhancement
                sr_img_np, _ = model.enhance(img_np, outscale=scale)
                sr_img = Image.fromarray(sr_img_np)
                upscaled_images.append(sr_img)
                
                # Save to ZIP buffer
                img_byte_arr = io.BytesIO()
                sr_img.save(img_byte_arr, format='PNG')
                zip_file.writestr(f"upscaled_{filename.split('.')[0]}.png", img_byte_arr.getvalue())

                del img_np, sr_img_np
                if is_cuda:
                    torch.cuda.empty_cache()

            except Exception as e:
                print(f"Error processing {filename}: {e}")
                continue

    zip_buffer.seek(0)
    zip_path = "upscaled_results.zip"
    with open(zip_path, "wb") as f:
        f.write(zip_buffer.getvalue())
    
    return zip_path, upscaled_images

# --- UI Setup with Gradio ---
with gr.Blocks() as demo:
    gr.Markdown("### 🛰️ SVNIT/ISRO Satellite & High-Res Image Upscaler")
    gr.Markdown(f"Current execution device: **{device.type.upper()}**")
    
    # Kernel Verification Alert
    if ".venv" not in sys.executable.lower():
        gr.Markdown("> [!CAUTION]\n> **WRONG KERNEL DETECTED!** Your notebook is using Global Python, not the project `.venv`. \n> \n> **To fix:** Click the kernel name (top-right of VS Code) → 'Select Kernel' → 'Python Environments' → Choose **.venv**.")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(file_count="multiple", label="Select Images to Upscale")
            scale_input = gr.Radio([2, 4, 8], label="Upscale Factor", value=4)
            btn = gr.Button("🚀 Start Upscaling", variant="primary")
        with gr.Column():
            zip_output = gr.File(label="Download ZIP of Results")
            gallery_output = gr.Gallery(label="Preview Results", columns=2)

    btn.click(batch_upscale, [file_input, scale_input], [zip_output, gallery_output])

if __name__ == "__main__":
    demo.launch(theme=gr.themes.Soft())